<a href="https://colab.research.google.com/github/acorrea79/techchallenge-fase2-pipeline-alfabetizacao/blob/main/notebooks/pipeline_alfabetizacao_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TechChallenge Fase 2 - Pipeline Híbrido para Análise da Alfabetização no Brasil

Projeto desenvolvido para a FIAP, turma 1IAST.

Integrante:

- Andre Correa Luis Vilas Boas

## Objetivo do notebook

Este notebook centraliza a execução acadêmica da pipeline de dados para análise da alfabetização no Brasil.

A solução utiliza:

- GCP BigQuery Sandbox;
- Google Colab;
- Python;
- Pandas;
- arquitetura Medalhão com camadas Bronze, Silver e Gold;
- ingestão Batch;
- simulação de Streaming;
- validações de qualidade;
- logs de monitoramento.

Este notebook faz parte da entrega do TechChallenge Fase 2.

## Instalação das bibliotecas

In [1]:
!pip install -q pandas numpy pyarrow google-cloud-bigquery db-dtypes

## Imports básicos do projeto

In [2]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

from google.colab import auth
from google.cloud import bigquery

## Configurações do projeto

In [3]:
# Configurações principais do projeto

PROJECT_ID = "fiap-techchallenge-fase2"
DATASET_ID = "basedosdados.br_inep_avaliacao_alfabetizacao"

BASE_PATH = Path("/content/techchallenge-fase2-pipeline-alfabetizacao")

BRONZE_PATH = BASE_PATH / "data" / "bronze"
BRONZE_BATCH_PATH = BRONZE_PATH / "batch"
BRONZE_STREAMING_PATH = BRONZE_PATH / "streaming"

SILVER_PATH = BASE_PATH / "data" / "silver"
GOLD_PATH = BASE_PATH / "data" / "gold"
LOGS_PATH = BASE_PATH / "logs"

TABLES = {
    "alunos": f"{DATASET_ID}.alunos",
    "dicionario": f"{DATASET_ID}.dicionario",
    "meta_alfabetizacao_brasil": f"{DATASET_ID}.meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": f"{DATASET_ID}.meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": f"{DATASET_ID}.meta_alfabetizacao_municipio",
    "municipio": f"{DATASET_ID}.municipio",
    "uf": f"{DATASET_ID}.uf",
}

print("Configuração carregada com sucesso.")
print(f"Projeto GCP: {PROJECT_ID}")
print(f"Dataset principal: {DATASET_ID}")

Configuração carregada com sucesso.
Projeto GCP: fiap-techchallenge-fase2
Dataset principal: basedosdados.br_inep_avaliacao_alfabetizacao


## Criação das pastas da pipeline

In [4]:
# Criação da estrutura de pastas no ambiente Colab

paths = [
    BRONZE_BATCH_PATH,
    BRONZE_STREAMING_PATH,
    SILVER_PATH,
    GOLD_PATH,
    LOGS_PATH,
]

for path in paths:
    path.mkdir(parents=True, exist_ok=True)

print("Estrutura de pastas criada no Colab:")
for path in paths:
    print(f"- {path}")

Estrutura de pastas criada no Colab:
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold
- /content/techchallenge-fase2-pipeline-alfabetizacao/logs


## Função simples de log

In [5]:
# Função de log para monitoramento simples da pipeline

def write_log(message: str, level: str = "INFO"):
    LOGS_PATH.mkdir(parents=True, exist_ok=True)

    log_file = LOGS_PATH / "pipeline_execution.log"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    log_line = f"{timestamp} | {level} | {message}"

    with open(log_file, "a", encoding="utf-8") as file:
        file.write(log_line + "\n")

    print(log_line)


write_log("Notebook principal iniciado.")

2026-07-01 01:57:54 | INFO | Notebook principal iniciado.


## Autenticação no Google

In [6]:
# Autenticação no Google
# Será aberta uma janela para autorizar o acesso com sua conta Google.

auth.authenticate_user()

write_log("Autenticação no Google realizada com sucesso.")

2026-07-01 01:57:54 | INFO | Autenticação no Google realizada com sucesso.


## Criação do cliente BigQuery

In [7]:
# Criação do cliente BigQuery

client = bigquery.Client(project=PROJECT_ID)

write_log("Cliente BigQuery criado com sucesso.")

2026-07-01 01:57:54 | INFO | Cliente BigQuery criado com sucesso.


##Teste de acesso ao BigQuery

In [8]:
# Teste simples de acesso ao BigQuery Sandbox

query = """
SELECT
  'BigQuery Sandbox acessado com sucesso' AS status
"""

df_test = client.query(query).to_dataframe(create_bqstorage_client=False)

display(df_test)

write_log("Consulta de teste executada com sucesso.")

,status
0,BigQuery Sandbox acessado com sucesso


2026-07-01 01:57:55 | INFO | Consulta de teste executada com sucesso.


# Listar tabelas do dataset principal

In [9]:
!gcloud auth list
!gcloud config set project fiap-techchallenge-fase2
!gcloud config get-value project

query_tables = """
SELECT
  table_catalog,
  table_schema,
  table_name,
  table_type
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLES`
ORDER BY
  table_name
"""

df_tables = client.query(query_tables).to_dataframe(create_bqstorage_client=False)

display(df_tables)

write_log(f"Listagem de tabelas executada. Total de tabelas encontradas: {len(df_tables)}")

      Credentialed Accounts
ACTIVE  ACCOUNT
*       gabrielhcorrea27@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`

Updated property [core/project].
fiap-techchallenge-fase2


,table_catalog,table_schema,table_name,table_type
0,basedosdados,br_inep_avaliacao_alfabetizacao,alunos,BASE TABLE
1,basedosdados,br_inep_avaliacao_alfabetizacao,dicionario,BASE TABLE
2,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_brasil,BASE TABLE
3,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_municipio,BASE TABLE
4,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_uf,BASE TABLE
5,basedosdados,br_inep_avaliacao_alfabetizacao,municipio,BASE TABLE
6,basedosdados,br_inep_avaliacao_alfabetizacao,uf,BASE TABLE


2026-07-01 01:58:02 | INFO | Listagem de tabelas executada. Total de tabelas encontradas: 7


## Listar colunas do dataset principal

In [10]:
# Listagem das colunas das tabelas do dataset principal

query_columns = """
SELECT
  table_name,
  column_name,
  data_type,
  is_nullable,
  ordinal_position
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.COLUMNS`
ORDER BY
  table_name,
  ordinal_position
"""

df_columns = client.query(query_columns).to_dataframe(create_bqstorage_client=False)

display(df_columns)

write_log(f"Schema carregado com sucesso. Total de colunas encontradas: {len(df_columns)}")

,table_name,column_name,data_type,is_nullable,ordinal_position
0,alunos,ano,INT64,YES,1
1,alunos,id_municipio,STRING,YES,2
2,alunos,id_escola,STRING,YES,3
3,alunos,id_aluno,STRING,YES,4
4,alunos,caderno,STRING,YES,5
...,...,...,...,...,...
78,uf,proporcao_aluno_nivel_4,FLOAT64,YES,11
79,uf,proporcao_aluno_nivel_5,FLOAT64,YES,12
80,uf,proporcao_aluno_nivel_6,FLOAT64,YES,13
81,uf,proporcao_aluno_nivel_7,FLOAT64,YES,14


2026-07-01 01:58:04 | INFO | Schema carregado com sucesso. Total de colunas encontradas: 83


## Volume das tabelas

In [11]:
# Consulta de volume das tabelas
# Esta etapa apoia decisões de FinOps.
# Tentamos primeiro INFORMATION_SCHEMA.TABLE_STORAGE.
# Caso falhe por restrição de localização/metadados, usamos __TABLES__ como alternativa.

query_storage_information_schema = """
SELECT
  table_name,
  total_rows,
  ROUND(total_logical_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLE_STORAGE`
ORDER BY
  total_logical_bytes DESC
"""

query_storage_legacy = """
SELECT
  table_id AS table_name,
  row_count AS total_rows,
  ROUND(size_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.__TABLES__`
ORDER BY
  size_bytes DESC
"""

try:
    df_storage = client.query(query_storage_information_schema).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "INFORMATION_SCHEMA.TABLE_STORAGE"
    write_log("Consulta de volume executada via INFORMATION_SCHEMA.TABLE_STORAGE.")

except Exception as error:
    write_log(
        f"Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. "
        f"Usando alternativa __TABLES__. Erro: {str(error)[:200]}",
        level="WARNING"
    )

    df_storage = client.query(query_storage_legacy).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "__TABLES__"
    write_log("Consulta alternativa de volume executada via __TABLES__.")

display(df_storage)

print(f"Fonte usada para volume das tabelas: {storage_source}")

2026-07-01 01:58:04 | WARNING | Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. Usando alternativa __TABLES__. Erro: 404 Not found: Dataset basedosdados:br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA was not found in location US; reason: notFound, message: Not found: Dataset basedosdados:br_inep_avaliacao_alfabe
2026-07-01 01:58:05 | INFO | Consulta alternativa de volume executada via __TABLES__.


,table_name,total_rows,tamanho_mb
0,alunos,3867999,256.10
1,municipio,23995,1.75
2,meta_alfabetizacao_municipio,10704,1.10
3,uf,145,0.01
4,meta_alfabetizacao_uf,81,0.01
5,dicionario,27,0.00
6,meta_alfabetizacao_brasil,3,0.00


Fonte usada para volume das tabelas: __TABLES__


## Validação das tabelas obrigatórias

In [12]:
# Validação das tabelas obrigatórias do desafio

expected_tables = {
    "alunos",
    "dicionario",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
}

found_tables = set(df_tables["table_name"].tolist())

missing_tables = expected_tables - found_tables
extra_tables = found_tables - expected_tables

validation_result = {
    "expected_tables": sorted(list(expected_tables)),
    "found_tables": sorted(list(found_tables)),
    "missing_tables": sorted(list(missing_tables)),
    "extra_tables": sorted(list(extra_tables)),
    "status": "approved" if not missing_tables else "failed"
}

display(pd.DataFrame([validation_result]))

if missing_tables:
    write_log(f"Tabelas obrigatórias ausentes: {missing_tables}", level="ERROR")
else:
    write_log("Todas as tabelas obrigatórias foram encontradas.")

,expected_tables,found_tables,missing_tables,extra_tables,status
0,"[alunos, dicionario, meta_alfabetizacao_brasil...","[alunos, dicionario, meta_alfabetizacao_brasil...",[],[],approved


2026-07-01 01:58:05 | INFO | Todas as tabelas obrigatórias foram encontradas.


## Salvar metadados no ambiente do Colab

In [13]:
# Salvando metadados de descoberta em arquivos locais do Colab

metadata_path = BRONZE_BATCH_PATH / "metadata"
metadata_path.mkdir(parents=True, exist_ok=True)

df_tables.to_csv(metadata_path / "dataset_tables.csv", index=False)
df_columns.to_csv(metadata_path / "dataset_columns.csv", index=False)
df_storage.to_csv(metadata_path / "dataset_storage.csv", index=False)

with open(metadata_path / "table_validation.json", "w", encoding="utf-8") as file:
    json.dump(validation_result, file, ensure_ascii=False, indent=4)

write_log("Metadados do dataset salvos na camada Bronze/metadata.")

2026-07-01 01:58:05 | INFO | Metadados do dataset salvos na camada Bronze/metadata.


## Verificar arquivos gerados

In [14]:
# Verificação dos arquivos gerados

for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(str(BASE_PATH), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

techchallenge-fase2-pipeline-alfabetizacao/
  data/
    bronze/
      batch/
        metadata/
          dataset_columns.csv
          table_validation.json
          dataset_tables.csv
          dataset_storage.csv
      streaming/
    silver/
    gold/
  logs/
    pipeline_execution.log
